In [ ]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, START, END

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

class AgentState(TypedDict):
    task: str
    worker_task: str
    code: str
    verification_result: str
    verification_status: str
    attempts: int
    max_attempts: int
    supervisor_decision: str
    final_answer: str

class supervisor_decision(BaseModel):
    decision: Literal["delegate", "finish"] = Field(description = "delegate = worker must continue working. Finish = task has been verified successfully.")
    instruction: str = Field(description = "Detailed instruction to the worker. If the previous attempt failed exlpain what needs to be fixed.")

def supervisor(state: AgentState):
    attempts = state["attempts"]
    if attempts >= state["max_attempts"]:
        return {"supervisor_decision" : "finish", "final_answer" : f"The task could not be varified within {state=["max_attempts"]} attempts."}
    prompt = f""" You are the supervisor of an autonomous sofware development team.
                  USER TASK: {state["task"]}
                  CURRENT CODE: {state["code"]}
                  PREVIOUS VARIFICATION STATUS: {state["varification_status"]}
                  PREVIOUS VARIFICATION RESULT: {state["varification_result"]}
                  CURRENT ATTEMPT: {attempts}
                  Your job is to decide whether the worker needs to continue.
                  IMPORTANT:
                  1. Do not trust the worker merely saying that the code works.
                  2. The varification result is authoritative.
                  3. If the varification failed, delegate the problem back to the worker.
                  4. If the varification passed and the implementation satisfies the original task, finish.
                  5. If there is still works to do, explain precisely what the worker should do next.
                  You may delegate repeatedly."""
    decision = model.with_structured_output(supervisor_decision()).invoke([SystemMessage(content=prompt)])
    return {"supervisor_decision": decision.decision, "worker_task": decision.instruction}

def worker(state: AgentState):
    new_attempt = state["attempts"] + 1
    prompt = f""" You are the software developer in an autonomous agent system.
                  ORIGINAL USER TASK: {state["task"]}
                  SUPERVISOR INSTRUCTION: {state["worker_task"]}
                  CURRENT CODE: {state["code"]}
                  PREVIOUS VARIFICATION: {state["varification_result"]}
                  Write of fix the implementation. The implementation must satisfy the original task. Previous varification failures are important.
                  Do not simply explain what should be changed. Produce the completed corrected source code. Return only the complete python source code."""
    response = model.invoke([SystemMessage(content=prompt)])
    return {"code": response.content, "attempts": new_attempt, "varification_status": "NOT_VARIFIED"}

def varifier(state: AgentState):
    varification_prompt = f"""You are a strict software varifier.
                              ORIGINAL TASK: {state["task"]}
                              IMPLEMENTATION: {state["code"]}
                              Determine whether the implementation satisfies the task.
                              Check:
                              1. Is the requested functionality implemented?
                              2. Is the python code syntatically valid?
                              3. Does the implementation logically work?
                              4. Are edge cases reasonably handled?
                              5. Does it satisfies the original requirement?
                              Return:
                              PASS or FAIL followed by detailed evidence.
                              Do not assume the implementation works merely because the developer claims it works."""
    result = model.invoke([SystemMessage(content=varification_prompt)]).content
    return {"varification_result": result, "varification_status": "PASS" if result.strip().upper().startswith("PASS") else "FAIL"}

def supervisor_router(state: AgentState) -> Literal["worker", "end"]:
    return "end" if state["supervisor_decision"] == "finish" else "worker"

